# 02 · Baselines B1 (PatchCore), B3 (detector + template report), B2 (zero-shot VLMs)
All thresholds are chosen on **val**. Do not run `--split test` here.

## 0. Session setup (run every session)
Upload the project folder (code, `configs/`, `data/manifests/`, `data/licenses/`; **not** `data/raw`, `.venv`) to
`MyDrive/InspectEdge`. Runtime → Change runtime type → **GPU** (T4 is enough for 500M; L4/A100 for 3B).

In [ ]:
import os, json, subprocess
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/InspectEdge'   # persistent: code, manifests, reports, checkpoints
WORK  = '/content/InspectEdge'                 # fast local disk: code copy + raw images
os.makedirs(WORK, exist_ok=True)
!rsync -a --exclude 'data/raw' --exclude '.venv' --exclude 'checkpoints' {DRIVE}/ {WORK}/
%cd {WORK}
!pip -q install -r requirements-colab.txt
!pip -q install -e . --no-deps
os.environ['INSPECTEDGE_ROOT'] = WORK
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, transformers, peft
print('torch', torch.__version__, 'cuda', torch.version.cuda, '| transformers', transformers.__version__, '| peft', peft.__version__)
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> GPU'

In [ ]:
# Data: official VisA release (CC BY 4.0). Verified by SHA-256 before use.
VISA_SHA = '2eb8690c803ab37de0324772964100169ec8ba1fa3f7e94291c9ca673f40f362'
if not os.path.exists('data/raw/visa/pcb1'):
    os.makedirs('data/raw/_dl', exist_ok=True); os.makedirs('data/raw/visa', exist_ok=True)
    !wget -q -O data/raw/_dl/VisA_20220922.tar https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar
    h = subprocess.run(['sha256sum', 'data/raw/_dl/VisA_20220922.tar'], capture_output=True, text=True).stdout.split()[0]
    assert h == VISA_SHA, f'VisA archive checksum mismatch: {h}'
    !tar -xf data/raw/_dl/VisA_20220922.tar -C data/raw/visa && rm data/raw/_dl/VisA_20220922.tar
!ls data/raw/visa

In [ ]:
# Manifest: rebuilt deterministically each session; fails loudly if the locked split changed.
!python -m inspectedge.data.build_manifest

## B1 · PatchCore on the official VisA 1-class split (+ corruption suite, severity 3)
~3–5 min/category on T4.

In [ ]:
!python scripts/baseline_patchcore.py --corrupt 2>&1 | grep -v Warning

In [ ]:
r = json.load(open('reports/baseline_patchcore/metrics.json'))
import pandas as pd
cats = [c for c in r if not c.startswith('_')]
pd.DataFrame({c: {k: r[c][k] for k in ['image_auroc','image_auprc','image_f1_max','pixel_auroc','aupro_0.3']} for c in cats}).T.round(3)

In [ ]:
rob = pd.DataFrame({c: {f: v['image_auroc'] for f, v in r[c]['robustness_sev3'].items()} for c in cats}).T
rob.insert(0, 'clean', [r[c]['image_auroc'] for c in cats]); rob.round(3)

## B3 · Detector + template report on the VLM val split (freezes per-category thresholds)

In [ ]:
!python scripts/b3_detector_report.py --split val 2>&1 | grep -v Warning | tail -25

## B2 · Zero-shot small VLMs (same prompt/schema as fine-tuning)

In [ ]:
for m in ['HuggingFaceTB/SmolVLM-256M-Instruct', 'HuggingFaceTB/SmolVLM-500M-Instruct']:
    !python scripts/eval_vlm.py --model {m} --split val --per-cell 10 2>&1 | grep -v Warning | tail -30

Optional quality challenger (needs L4/A100, 4-bit; research comparison only — not deployable on phone):

In [ ]:
# !python scripts/eval_vlm.py --model Qwen/Qwen2.5-VL-3B-Instruct --load-4bit --split val --per-cell 5

In [ ]:
!python scripts/compare_and_gate.py --split val --system B3=reports/b3_detector_report/val_predictions.jsonl --system B2_500M=reports/vlm_eval/SmolVLM-500M-Instruct_val_predictions.jsonl

In [ ]:
# Persist outputs to Drive (run at the end and after long steps)
!rsync -aR reports data/manifests {DRIVE}/
print('synced to', DRIVE)